# Llama 3.1 Finetuning

## Setup

In [ ]:
import torch
from pathlib import Path

from chud.preprocess import DataProcessor, load_posts
from chud.finetuner import (
    LlamaFineTuner, 
    LoRAParams, 
    TrainingParams, 
    generate_text
)

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Configuration

In [ ]:
# =============================================================================
# GENERAL CONFIGURATION
# =============================================================================

DATA_FILE = "./data/posts.json"
TRAINING_MODE = "completion"
MODEL_NAME = "meta-llama/Llama-3.1-8B"
OUTPUT_DIR = "./output/llama-4chan-finetuned"
USE_4BIT = True
USE_8BIT = False

# =============================================================================
# LORA CONFIGURATION
# =============================================================================

LORA_R = 16          
LORA_ALPHA = 32      
LORA_DROPOUT = 0.05  

# =============================================================================
# TRAINING CONFIGURATION
# =============================================================================

EPOCHS = 3
BATCH_SIZE = 4             
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 512        
GRADIENT_ACCUMULATION = 4 

## Load and Prepare Data

In [ ]:
# Load scraped posts
posts = load_posts(DATA_FILE)
print(f"Loaded {len(posts)} posts")

processor = DataProcessor(min_length=20, max_length=2000)
dataset = processor.create_dataset(posts, mode=TRAINING_MODE)
print(f"Training examples: {len(dataset)}")
print(f"Training mode: {TRAINING_MODE}")

## Initialize the Finetuner

In [ ]:
finetuner = LlamaFineTuner(
    model_name=MODEL_NAME,
    output_dir=OUTPUT_DIR,
    use_4bit=USE_4BIT,
    use_8bit=USE_8BIT
)

print(f"Model: {finetuner.model_name}")
print(f"Output: {finetuner.output_dir}")
print(f"Quantization: {'4-bit' if USE_4BIT else '8-bit' if USE_8BIT else 'None'}")

## Load the Base Model

In [ ]:
%%time
finetuner.load_model()

## Setup LoRA

In [ ]:
# Configure LoRA parameters
lora_params = LoRAParams(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT
)

print(f"LoRA config:")
print(f"  Rank (r): {lora_params.r}")
print(f"  Alpha: {lora_params.lora_alpha}")
print(f"  Dropout: {lora_params.lora_dropout}")
print(f"  Target modules: {lora_params.target_modules}")

In [ ]:
# Apply LoRA to the model
finetuner.setup_lora(lora_params)

## Train the Model

In [ ]:
# Configure training parameters
training_params = TrainingParams(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    max_seq_length=MAX_SEQ_LENGTH,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION
)

print("Training config:")
print(f"  Epochs: {training_params.epochs}")
print(f"  Batch size: {training_params.batch_size}")
print(f"  Gradient accumulation: {training_params.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_params.batch_size * training_params.gradient_accumulation_steps}")
print(f"  Learning rate: {training_params.learning_rate}")
print(f"  Max sequence length: {training_params.max_seq_length}")

In [ ]:
%%time
finetuner.train(dataset, training_params)

## Save the Model

In [ ]:
print(f"LoRA adapter saved to: {OUTPUT_DIR}")
print(f"\nFiles in output directory:")
for f in Path(OUTPUT_DIR).iterdir():
    size_mb = f.stat().st_size / 1e6 if f.is_file() else 0
    print(f"  {f.name} ({size_mb:.1f} MB)" if size_mb > 0 else f"  {f.name}/")

In [ ]:
# Optional: Merge LoRA weights into base model
# This creates a standalone model that doesn't need the adapter
# Warning: This requires more disk space

MERGE_WEIGHTS = False

if MERGE_WEIGHTS:
    merged_dir = finetuner.merge_and_save()
    print(f"Merged model saved to: {merged_dir}")